# Brecha en acceso a fuente de agua mejorada — Colombia 2018-2025

---

**Materia:** Storytelling con datos  
**Fuente:** DANE — Encuesta Nacional de Calidad de Vida (ECV) 2018-2025  
**Herramientas:** Python · Dash · Plotly · GeoPandas

---

## Mensaje clave

> *En Colombia, vivir en el campo sigue significando vivir sin agua segura. La brecha entre
> cabeceras municipales y centros poblados rurales en el indicador 'Sin acceso a fuente de
> agua mejorada' del IPM supera los 30 puntos porcentuales a nivel nacional en 2025,
> y alcanza extremos de mas de 90 pp en departamentos como Vaupes y Vichada.*

## Contexto del indicador

El **Indice de Pobreza Multidimensional (IPM)** del DANE incluye 15 indicadores agrupados
en 5 dimensiones. El indicador **'Sin acceso a fuente de agua mejorada'** pertenece a la
dimension de **Vivienda** (peso: 4% del IPM total). Se activa cuando un hogar **no** cuenta
con ninguna de las siguientes fuentes: acueducto, agua lluvia recolectada en tanque, pila
publica, pozo con bomba, pozo protegido o agua embotellada.

## Diseño de la visualizacion

### Tipo de grafico
**Mapa coropletico interactivo (Dash + Plotly)** con panel desplegable (modal) por
departamento. Se eligio un solo grafico principal (mapa) con elementos emergentes
(modal con serie de tiempo + parrafo reflexivo) para cumplir la restriccion de
'una sola visualizacion' sin sacrificar la narrativa temporal.

### Variable codificada en color
El mapa colorea cada departamento segun la **brecha en puntos porcentuales**
(`Rural - Cabeceras`) para el ano seleccionado en el slider:
- **Ambar dorado** (`#FFB300`): brecha baja — mayor igualdad territorial
- **Azul oscuro** (`#0a1628`): brecha alta — desigualdad estructural profunda

Esta paleta sigue los principios del curso:
- Evita el efecto semaforo (no usa rojo/verde)
- Usa el contraste frio-calido (azul profundo vs ambar dorado)
- No estigmatiza visualmente las zonas de mayor privacion
- Los extremos saturados atraen la atencion (elemento preatentivo)

### Elementos preatentivos usados
1. **Color**: codificacion de la brecha en el mapa (dorado=menor, azul=mayor)
2. **Tamaño tipografico**: titulo grande (26px, 900 weight) vs subtitulo (14px)
3. **Color en texto**: palabras 'Cabeceras' en ambar y 'Rural disperso' en azul oscuro
   dentro del parrafo del modal — la leyenda esta inmersa en el texto
4. **Numero destacado**: brecha nacional en 38px, color azul oscuro, en el card del encabezado
5. **Borde lateral**: linea azul oscura en el parrafo de conclusion
6. **Sombreado de area**: fill translucido entre las dos lineas del grafico de brecha

### Temporalidad
- El **slider** controla el ano del mapa (snapshot territorial)
- El **modal** siempre muestra la serie completa 2018-2025 para el departamento clickeado
- El **panel nacional** muestra la serie completa con una linea punteada marcando el ano seleccionado

---

## Estructura del codigo

```
agua_brecha_app.py
├── CONFIGURACION GLOBAL     — rutas, paleta de colores, diccionarios
├── load_agua_data()         — carga y limpieza del Excel DANE
├── load_geodata()           — carga shapefile y union con datos
├── compute_national()       — promedios nacionales por ano
├── build_map()              — figura choropleth
├── build_national_chart()   — figura lineas brecha nacional
├── build_dept_modal_chart() — figura lineas por departamento (modal)
├── generate_dept_paragraph()— parrafo HTML reflexivo con colores
├── LAYOUT Dash              — estructura HTML/Bootstrap del dashboard
├── CALLBACKS Dash           — logica de interactividad
└── export_static_html()     — exporta choropleth estatico a HTML
```

---


## Celda 1 — Instalacion de dependencias

Se instalan todas las librerias necesarias:
- **`dash`**: framework web reactivo para el dashboard interactivo
- **`dash-bootstrap-components`**: grid Bootstrap, modales y estilos
- **`plotly`**: graficos interactivos (mapa, lineas)
- **`geopandas`**: lectura del shapefile de departamentos
- **`openpyxl`**: lectura del Excel del DANE con celdas fusionadas


In [ ]:
# Instalacion de dependencias (ejecutar solo una vez)
!pip install dash dash-bootstrap-components plotly geopandas openpyxl pandas numpy

## Celda 2 — Imports y configuracion global

Se importan todas las librerias y se definen las constantes globales:
- Rutas a los archivos de datos
- Paleta de colores basada en psicologia del color
- Diccionarios de normalizacion de nombres de departamentos


In [ ]:
import os
import json
import zipfile
import unicodedata

import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, State
import dash_bootstrap_components as dbc

# ── Rutas de archivos ────────────────────────────────────────────────────────
EXCEL_PATH = "anexPMultidimensionalDepartamental2025.xlsx"
ZIP_PATH   = "MGN2024_DPTO_POLITICO.zip"    # shapefile DANE — mismo del mapa anterior
SHP_DIR    = "shp_departamentos"
YEARS      = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# ── Paleta de color ──────────────────────────────────────────────────────────
# Psicologia del color:
#   Ambar/dorado -> oportunidad, luz, menor privacion
#   Azul oscuro  -> profundidad del problema, seriedad, sin estigma social
COLOR_CABECERAS = "#FFB300"               # ambar — acceso urbano
COLOR_RURAL     = "#1a3a6b"               # azul oscuro — privacion rural
COLOR_FILL      = "rgba(26,58,107,0.15)"  # area de brecha — translucido
COLOR_PAGE_BG   = "#f7f5f0"               # fondo — neutro calido
COLOR_CARD_BG   = "#ffffff"
COLOR_TEXT_DARK = "#0a1628"
COLOR_TEXT_MID  = "#4a5568"

# Colorscale del mapa: brecha baja=dorado -> brecha alta=azul muy oscuro
MAP_COLORSCALE = [
    [0.00, "#FFB300"],   # brecha muy baja  (<5 pp)  -> ambar
    [0.08, "#c4a35a"],   # brecha baja                -> oro transicional
    [0.20, "#8a9ab5"],   # brecha media               -> gris azulado
    [0.40, "#4a7ab5"],   # brecha moderada            -> azul medio
    [0.65, "#1a3a6b"],   # brecha alta                -> azul oscuro
    [1.00, "#0a1628"],   # brecha extrema (>70 pp)    -> azul casi negro
]

# Normalizacion de nombres entre shapefile DANE (mayusculas) y Excel (titulo)
NOMBRE_MAP = {
    "BOGOTA, D.C."  : "Bogota",
    "BOGOTA D.C."   : "Bogota",
    "BOGOTA"        : "Bogota",
    "NORTE DE SANTANDER" : "Norte De Santander",
    "VALLE DEL CAUCA"    : "Valle Del Cauca",
    "SAN ANDRES, PROVIDENCIA Y SANTA CATALINA": "San Andres",
    "ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA": "San Andres",
    "QUINDIIO" : "Quindio",
    "QUINDIO"  : "Quindio",
    "NARINO"   : "Narino",
    "CHOCO"    : "Choco",
}

print("Configuracion cargada OK")

## Celda 3 — Funciones de normalizacion y carga de datos

### Estrategia de limpieza del Excel

La hoja `IPM_Indicadores_Departamento` tiene:
- 12 filas decorativas antes de los datos reales
- Fila 11 (0-indexed): anos (2018...2025)
- Fila 12 (0-indexed): encabezado de columnas (Departamento, Variable, Total, Cabeceras, Rural x 8)
- Fila 13+: datos, con el nombre del departamento solo en la primera fila del grupo
  (las demas filas del mismo departamento tienen NaN en col 0 — celdas fusionadas)

La limpieza:
1. Salta las 12 primeras filas
2. Asigna nombres de columna manualmente
3. Rellena hacia adelante la columna Departamento
4. Filtra solo la variable de interes
5. Calcula la brecha por ano = Rural - Cabeceras


In [ ]:
def normalize_name(name: str) -> str:
    """Normalize department name: remove accents, strip, title case."""
    name = str(name).strip()
    nfkd = unicodedata.normalize("NFD", name)
    no_accent = "".join(c for c in nfkd if not unicodedata.combining(c))
    return no_accent.strip().title()


def load_agua_data() -> pd.DataFrame:
    """
    Load 'Sin acceso a fuente de agua mejorada' from DANE Excel.
    Returns DataFrame with Cabeceras_YYYY, Rural_YYYY, Brecha_YYYY per year.
    """
    raw = pd.read_excel(
        EXCEL_PATH,
        sheet_name="IPM_Indicadores_Departamento ",  # nota: espacio al final del nombre
        header=None,
        skiprows=12,
    )

    # Construccion manual de nombres de columnas
    col_names = ["Departamento", "Variable"]
    for y in YEARS:
        col_names += [f"Total_{y}", f"Cabeceras_{y}", f"Rural_{y}"]
    raw.columns = col_names

    # Forward-fill: las celdas fusionadas del Excel dejan NaN en filas del mismo dpto
    raw["Departamento"] = raw["Departamento"].ffill()

    # Filtro al indicador de agua
    agua = raw[raw["Variable"] == "Sin acceso a fuente de agua mejorada"].copy()
    agua = agua[agua["Departamento"] != "Departamento"].reset_index(drop=True)

    # Conversion a numericos
    for y in YEARS:
        for domain in ["Total", "Cabeceras", "Rural"]:
            agua[f"{domain}_{y}"] = pd.to_numeric(agua[f"{domain}_{y}"], errors="coerce")

    # Calculo de brecha = Rural - Cabeceras por ano
    for y in YEARS:
        agua[f"Brecha_{y}"] = agua[f"Rural_{y}"] - agua[f"Cabeceras_{y}"]

    # Normalizacion de nombres para join con shapefile
    agua["Departamento_norm"] = agua["Departamento"].apply(normalize_name)
    agua["Departamento_norm"] = agua["Departamento_norm"].replace(
        {"Bogota, D.C.": "Bogota", "Bogota D.C.": "Bogota",
         "San Andres, Providencia Y Santa Catalina": "San Andres"}
    )
    return agua


def load_geodata(agua: pd.DataFrame):
    """
    Load and merge Colombia department shapefile with agua DataFrame.
    Requires MGN2024_DPTO_POLITICO.zip in the working directory.
    """
    # Extraccion del shapefile si no existe
    if not os.path.exists(SHP_DIR):
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall(SHP_DIR)

    shp_files = [f for f in os.listdir(SHP_DIR) if f.endswith(".shp")]
    assert shp_files, "No se encontro archivo .shp. Verifica MGN2024_DPTO_POLITICO.zip"
    gdf = gpd.read_file(os.path.join(SHP_DIR, shp_files[0]))

    # Normalizacion del nombre en el shapefile
    name_col = "DeNombre" if "DeNombre" in gdf.columns else gdf.columns[1]
    gdf["dpto_norm"] = gdf[name_col].apply(
        lambda x: normalize_name(NOMBRE_MAP.get(str(x).strip().upper(), str(x)))
    )

    # Proyeccion WGS84 requerida por Plotly
    if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    # Union con datos de agua
    merged = gdf.merge(agua, left_on="dpto_norm", right_on="Departamento_norm", how="left")
    return merged


def compute_national(agua: pd.DataFrame) -> dict:
    """Compute national averages (mean across 33 departments) for each year."""
    nacional = {}
    for y in YEARS:
        nacional[y] = {
            "cabeceras": round(float(agua[f"Cabeceras_{y}"].mean()), 1),
            "rural":     round(float(agua[f"Rural_{y}"].mean()), 1),
            "brecha":    round(float(agua[f"Brecha_{y}"].mean()), 1),
        }
    return nacional


# Carga inicial de datos
agua = load_agua_data()
gdf  = load_geodata(agua)
nac  = compute_national(agua)

print(f"Departamentos cargados: {len(agua)}")
print(f"Brecha nacional 2025: {nac[2025]['brecha']} pp")
print(f"Rural 2025: {nac[2025]['rural']}% | Cabeceras 2025: {nac[2025]['cabeceras']}%")
print(f"Top 3 brechas 2025:")
print(agua[['Departamento','Brecha_2025']].sort_values('Brecha_2025', ascending=False).head(3).to_string(index=False))

## Celda 4 — Funciones de construccion de graficos

Se definen tres funciones de visualizacion:

1. **`build_map()`**: mapa coropletico con color = brecha del ano seleccionado
2. **`build_national_chart()`**: lineas Cabeceras vs Rural promedio nacional con area sombreada
3. **`build_dept_modal_chart()`**: lineas por departamento para el modal desplegable
4. **`generate_dept_paragraph()`**: parrafo reflexivo con palabras coloreadas como leyenda inmersa


In [ ]:
def build_map(gdf, year: int, selected_dpto: str = None) -> go.Figure:
    """
    Build choropleth map encoding the gap (Rural - Cabeceras) as color.
    
    Color encoding (pre-attentive element):
      Amber/gold (#FFB300) = low gap = more equitable territory
      Dark blue  (#0a1628) = high gap = structural rural inequality
    """
    brecha_col = f"Brecha_{year}"
    geojson    = json.loads(gdf.to_json())
    ids        = list(range(len(gdf)))

    # Texto de hover con colores codificados
    hover_texts = []
    for _, row in gdf.iterrows():
        dpto = row.get("Departamento", row.get("dpto_norm", ""))
        cab  = row.get(f"Cabeceras_{year}", np.nan)
        rur  = row.get(f"Rural_{year}", np.nan)
        bre  = row.get(brecha_col, np.nan)
        if pd.isna(bre):
            hover_texts.append(f"<b>{dpto}</b><br>Sin datos")
        else:
            hover_texts.append(
                f"<b>{dpto}</b><br>"
                f"<span style='color:{COLOR_CABECERAS}'>&#9632;</span> Cabeceras: {cab:.1f}%<br>"
                f"<span style='color:{COLOR_RURAL}'>&#9632;</span> Rural disperso: {rur:.1f}%<br>"
                f"<b>Brecha: {bre:.1f} pp</b><br>"
                f"<i>Click para ver la serie completa</i>"
            )

    # Resaltar departamento seleccionado (borde dorado)
    line_widths = []
    line_colors = []
    for _, row in gdf.iterrows():
        dpto = row.get("Departamento", row.get("dpto_norm", ""))
        if selected_dpto and normalize_name(str(dpto)) == normalize_name(str(selected_dpto)):
            line_widths.append(3)
            line_colors.append("#FFB300")   # borde dorado = seleccionado
        else:
            line_widths.append(0.5)
            line_colors.append("#ffffff")

    brecha_vals = gdf[brecha_col].fillna(-1).tolist()
    valid_vals  = [v for v in brecha_vals if v >= 0]
    vmax = max(valid_vals) if valid_vals else 100

    fig = go.Figure(go.Choropleth(
        geojson=geojson,
        locations=ids,
        z=brecha_vals,
        featureidkey="id",
        colorscale=MAP_COLORSCALE,
        zmin=0,
        zmax=vmax,
        marker_line_width=line_widths,
        marker_line_color=line_colors,
        text=hover_texts,
        hovertemplate="%{text}<extra></extra>",
        colorbar=dict(
            title=dict(text="Brecha<br>(pp)",
                       font=dict(size=11, color=COLOR_TEXT_DARK)),
            tickfont=dict(size=10, color=COLOR_TEXT_MID),
            len=0.6, thickness=14, x=1.01, ticksuffix=" pp",
        ),
    ))
    fig.update_geos(fitbounds="locations", visible=False, bgcolor="rgba(0,0,0,0)")
    fig.update_layout(
        margin=dict(l=0, r=0, t=0, b=0),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        dragmode=False,
    )
    return fig


def build_national_chart(nacional: dict, selected_year: int) -> go.Figure:
    """
    Line chart of national Cabeceras vs Rural averages (2018-2025)
    with shaded area between lines (the gap). Dotted line for selected year.
    """
    cab_vals   = [nacional[y]["cabeceras"] for y in YEARS]
    rural_vals = [nacional[y]["rural"]     for y in YEARS]

    fig = go.Figure()
    # Area sombreada entre las dos lineas (la brecha)
    fig.add_trace(go.Scatter(
        x=YEARS + YEARS[::-1], y=rural_vals + cab_vals[::-1],
        fill="toself", fillcolor=COLOR_FILL,
        line=dict(color="rgba(0,0,0,0)"),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=YEARS, y=rural_vals,
        mode="lines+markers", name="Rural disperso",
        line=dict(color=COLOR_RURAL, width=2.5),
        marker=dict(size=7, color=COLOR_RURAL),
        hovertemplate="Rural %{x}: <b>%{y:.1f}%</b><extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=YEARS, y=cab_vals,
        mode="lines+markers", name="Cabeceras",
        line=dict(color=COLOR_CABECERAS, width=2.5),
        marker=dict(size=7, color=COLOR_CABECERAS),
        hovertemplate="Cabeceras %{x}: <b>%{y:.1f}%</b><extra></extra>",
    ))
    # Linea punteada del ano seleccionado
    sel_brecha = nacional[selected_year]["brecha"]
    fig.add_vline(
        x=selected_year, line_dash="dot",
        line_color="#888888", line_width=1.5,
        annotation_text=f"Brecha {selected_year}<br><b>{sel_brecha:.1f} pp</b>",
        annotation_position="top right",
        annotation_font_size=10,
        annotation_font_color=COLOR_TEXT_DARK,
    )
    fig.update_layout(
        margin=dict(l=30, r=10, t=10, b=30),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        xaxis=dict(tickvals=YEARS, tickfont=dict(size=9, color=COLOR_TEXT_MID),
                   showgrid=False, zeroline=False),
        yaxis=dict(title=dict(text="%", font=dict(size=9, color=COLOR_TEXT_MID)),
                   tickfont=dict(size=9, color=COLOR_TEXT_MID),
                   gridcolor="rgba(0,0,0,0.06)", zeroline=False),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                    font=dict(size=9, color=COLOR_TEXT_MID)),
        height=220,
    )
    return fig


def build_dept_modal_chart(dept_row: pd.Series, dept_name: str) -> go.Figure:
    """
    Time series chart for a specific department shown in the modal.
    Shows Cabeceras and Rural lines with annotations on each point,
    shaded area representing the gap, and annotation of maximum gap.
    """
    cab_vals   = [float(dept_row[f"Cabeceras_{y}"]) for y in YEARS]
    rural_vals = [float(dept_row[f"Rural_{y}"])     for y in YEARS]
    brecha_por_ano = [float(dept_row[f"Brecha_{y}"]) for y in YEARS]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=YEARS + YEARS[::-1], y=rural_vals + cab_vals[::-1],
        fill="toself", fillcolor=COLOR_FILL,
        line=dict(color="rgba(0,0,0,0)"),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=YEARS, y=rural_vals,
        mode="lines+markers+text", name="Rural disperso",
        line=dict(color=COLOR_RURAL, width=2.5, dash="dot"),
        marker=dict(size=8, color=COLOR_RURAL),
        text=[f"{v:.1f}%" for v in rural_vals],
        textposition="top center",
        textfont=dict(size=9, color=COLOR_RURAL),
        hovertemplate="Rural %{x}: <b>%{y:.1f}%</b><extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=YEARS, y=cab_vals,
        mode="lines+markers+text", name="Cabeceras",
        line=dict(color=COLOR_CABECERAS, width=2.5),
        marker=dict(size=8, color=COLOR_CABECERAS),
        text=[f"{v:.1f}%" for v in cab_vals],
        textposition="bottom center",
        textfont=dict(size=9, color="#b8860b"),
        hovertemplate="Cabeceras %{x}: <b>%{y:.1f}%</b><extra></extra>",
    ))
    # Anotacion de la brecha maxima
    max_idx   = int(np.argmax(brecha_por_ano))
    max_year  = YEARS[max_idx]
    midpoint  = (rural_vals[max_idx] + cab_vals[max_idx]) / 2
    fig.add_annotation(
        x=max_year, y=midpoint,
        text=f"Brecha maxima<br><b>{brecha_por_ano[max_idx]:.1f} pp</b>",
        showarrow=False, font=dict(size=10, color=COLOR_TEXT_DARK),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#cccccc", borderwidth=1, borderpad=4,
    )
    fig.update_layout(
        margin=dict(l=40, r=10, t=20, b=40),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        xaxis=dict(
            title=dict(text="Ano", font=dict(size=10, color=COLOR_TEXT_MID)),
            tickvals=YEARS, tickfont=dict(size=10, color=COLOR_TEXT_MID),
            showgrid=False, zeroline=False,
        ),
        yaxis=dict(
            title=dict(text="% hogares sin agua mejorada",
                       font=dict(size=10, color=COLOR_TEXT_MID)),
            tickfont=dict(size=10, color=COLOR_TEXT_MID),
            gridcolor="rgba(0,0,0,0.06)", ticksuffix="%", zeroline=False,
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.02,
                    xanchor="right", x=1, font=dict(size=10, color=COLOR_TEXT_MID)),
        height=320,
    )
    return fig


def generate_dept_paragraph(dept_row: pd.Series, dept_name: str) -> list:
    """
    Generate a reflexive HTML paragraph for the modal.
    Words 'Cabeceras' and 'Rural disperso' are colored inline
    (pre-attentive: legend embedded in text).
    """
    brecha_2025 = float(dept_row["Brecha_2025"])
    brecha_2018 = float(dept_row["Brecha_2018"])
    cab_2025    = float(dept_row["Cabeceras_2025"])
    rur_2025    = float(dept_row["Rural_2025"])
    cambio      = brecha_2025 - brecha_2018

    if brecha_2025 >= 50:
        nivel = "una de las brechas mas profundas del pais"
        reflexion = ("Esta cifra revela una deuda historica con las comunidades rurales: "
                     "nacer aqui implica una desventaja estructural severa en el acceso basico al agua.")
    elif brecha_2025 >= 30:
        nivel = "una brecha significativa, por encima del promedio nacional"
        reflexion = ("La diferencia entre vivir en la ciudad y en el campo es aqui palpable. "
                     "Reducir esta brecha requiere inversion focalizada en infraestructura hidrica rural.")
    elif brecha_2025 >= 15:
        nivel = "una brecha moderada pero que sigue siendo una deuda social"
        reflexion = ("Aunque ha mejorado, cada punto porcentual representa miles de hogares sin agua segura. "
                     "La equidad territorial aun no es una realidad plena para todos.")
    else:
        nivel = "una de las brechas mas bajas del pais"
        reflexion = ("Este departamento muestra que es posible reducir la desigualdad en acceso al agua. "
                     "Incluso brechas pequenas afectan a comunidades reales que merecen agua segura.")

    cambio_dir = f"redujo en {abs(cambio):.1f} pp" if cambio < 0 else f"aumento en {cambio:.1f} pp"

    return [
        html.P([
            f"En {dept_row['Departamento']}, el acceso a agua mejorada en ",
            html.Span("cabeceras", style={"color": COLOR_CABECERAS, "fontWeight": "bold"}),
            f" es del {cab_2025:.1f}%, mientras que en ",
            html.Span("centros poblados y rural disperso",
                      style={"color": COLOR_RURAL, "fontWeight": "bold"}),
            f" solo el {rur_2025:.1f}% tiene este servicio. Esto representa ",
            html.Strong(f"{nivel}: {brecha_2025:.1f} puntos porcentuales"),
            f" de diferencia. Entre 2018 y 2025, la brecha se {cambio_dir}.",
        ], style={"fontSize": "13px", "color": COLOR_TEXT_MID,
                  "lineHeight": "1.7", "marginBottom": "8px"}),
        html.P(reflexion,
               style={"fontSize": "12px", "color": COLOR_TEXT_MID,
                      "fontStyle": "italic", "lineHeight": "1.6"}),
    ]


print("Funciones de graficos definidas OK")

## Celda 5 — Layout del dashboard

Estructura de la pagina:

```
+----------------------------------------------------------+
| Encabezado: Titulo + Subtitulo + Card brecha nacional    |
+----------------------------------------------------------+
| Slider de año (2018-2025)                                |
+---------------------+------------------------------------+
| MAPA COROPLETICO    | PANEL BRECHA NACIONAL              |
| (col 8/12)          | grafico lineas 2018-2025           |
|                     | + leyenda + lectura del mapa       |
+---------------------+------------------------------------+
| Parrafo de reflexion / conclusion                        |
+----------------------------------------------------------+
| Fuentes                                                  |
+----------------------------------------------------------+
```

**Modal desplegable** (se abre al hacer click en el mapa):
```
+-------------------------------------------+
| Titulo: Nombre departamento (año 2018-2025)|
| Subtitulo: valores clave del departamento  |
| Grafico: lineas + area sombreada + valores |
| Parrafo: reflexivo con colores en palabras |
+-------------------------------------------+
```


In [ ]:
# ── Estilos reutilizables ────────────────────────────────────────────────────
STYLE_TITLE = {
    "fontSize": "26px", "fontWeight": "900",
    "color": COLOR_TEXT_DARK, "lineHeight": "1.2",
    "marginBottom": "6px", "letterSpacing": "-0.3px",
}
STYLE_SUBTITLE = {
    "fontSize": "14px", "color": COLOR_TEXT_MID,
    "fontStyle": "italic", "lineHeight": "1.5", "marginBottom": "0",
}
STYLE_LABEL = {
    "fontSize": "11px", "fontWeight": "700",
    "letterSpacing": "1.5px", "textTransform": "uppercase",
    "color": COLOR_TEXT_MID, "marginBottom": "4px",
}
STYLE_CARD = {
    "backgroundColor": COLOR_CARD_BG, "borderRadius": "10px",
    "padding": "16px", "boxShadow": "0 1px 6px rgba(0,0,0,0.07)",
}

app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])
app.title = "Brecha de Agua — Colombia"

app.layout = html.Div(
    style={"backgroundColor": COLOR_PAGE_BG, "minHeight": "100vh",
           "fontFamily": "Inter, Arial, sans-serif"},
    children=[

        # ENCABEZADO
        dbc.Container(fluid=True, style={"padding": "24px 32px 0"}, children=[
            dbc.Row([
                dbc.Col([
                    html.P("INDICADOR DE VIVIENDA — IPM COLOMBIA", style=STYLE_LABEL),
                    html.H1(
                        "El campo sigue sin agua: la desigualdad que divide a Colombia",
                        style=STYLE_TITLE,
                    ),
                    html.P(
                        "Mientras en las ciudades casi todos tienen acceso a una fuente de agua mejorada, "
                        "en los centros poblados y el rural disperso una fraccion significativa de hogares "
                        "sigue enfrentando esta privacion basica. Esta brecha no es geografica: es politica. "
                        "Explora como varia por departamento y como ha cambiado entre 2018 y 2025.",
                        style=STYLE_SUBTITLE,
                    ),
                ], md=9),
                dbc.Col([
                    html.Div(style={**STYLE_CARD, "textAlign": "center", "marginTop": "8px"}, children=[
                        html.P("BRECHA NACIONAL 2025", style={**STYLE_LABEL, "marginBottom": "2px"}),
                        html.P(f"{nac[2025]['brecha']:.1f} pp",
                               style={"fontSize": "38px", "fontWeight": "900",
                                      "color": COLOR_RURAL, "marginBottom": "0"}),
                        html.P(f"Rural: {nac[2025]['rural']:.1f}%  vs  Cabeceras: {nac[2025]['cabeceras']:.1f}%",
                               style={"fontSize": "11px", "color": COLOR_TEXT_MID}),
                    ]),
                ], md=3),
            ], className="align-items-center"),
        ]),

        html.Hr(style={"margin": "16px 32px", "borderColor": "rgba(0,0,0,0.1)"}),

        # CONTROLES
        dbc.Container(fluid=True, style={"padding": "0 32px"}, children=[
            dbc.Row([
                dbc.Col([
                    html.P("SELECCIONA EL ANO", style={**STYLE_LABEL, "marginBottom": "4px"}),
                    dcc.Slider(
                        id="year-slider", min=2018, max=2025, step=1, value=2025,
                        marks={y: str(y) for y in YEARS},
                        tooltip={"always_visible": False}, className="mb-1",
                    ),
                ], md=7),
                dbc.Col([
                    html.P(
                        [html.Span("Haz click",
                                  style={"color": COLOR_CABECERAS, "fontWeight": "700"}),
                         " en cualquier departamento del mapa para ver su serie temporal completa."],
                        style={"fontSize": "12px", "color": COLOR_TEXT_MID, "marginTop": "28px"},
                    ),
                ], md=5),
            ]),
        ]),

        # MAPA + PANEL NACIONAL
        dbc.Container(fluid=True, style={"padding": "8px 32px"}, children=[
            dbc.Row([
                dbc.Col([
                    html.Div(style=STYLE_CARD, children=[
                        dcc.Graph(
                            id="mapa-agua",
                            style={"height": "560px"},
                            config={"displayModeBar": False, "scrollZoom": False},
                        ),
                    ]),
                ], md=8),
                dbc.Col([
                    html.Div(style={**STYLE_CARD, "height": "100%"}, children=[
                        html.P("EVOLUCION NACIONAL DE LA BRECHA", style=STYLE_LABEL),
                        html.P(
                            [html.Span("Cabeceras",
                                       style={"color": COLOR_CABECERAS, "fontWeight": "700"}),
                             " vs ",
                             html.Span("Rural disperso",
                                       style={"color": COLOR_RURAL, "fontWeight": "700"}),
                             " — promedio de los 33 departamentos"],
                            style={"fontSize": "11px", "color": COLOR_TEXT_MID, "marginBottom": "4px"},
                        ),
                        dcc.Graph(id="chart-nacional", config={"displayModeBar": False}),
                        html.Hr(style={"margin": "10px 0", "borderColor": "rgba(0,0,0,0.08)"}),
                        html.P("QUE SIGNIFICA LA BRECHA",
                               style={**STYLE_LABEL, "marginTop": "6px"}),
                        html.P(
                            "El area sombreada representa los hogares que viven esa diferencia "
                            "cada dia: sin agua de red, sin pozo protegido, sin agua segura para beber. "
                            "Cada punto porcentual en la brecha representa miles de familias rurales.",
                            style={"fontSize": "11px", "color": COLOR_TEXT_MID,
                                   "lineHeight": "1.6", "fontStyle": "italic"},
                        ),
                        html.Hr(style={"margin": "10px 0", "borderColor": "rgba(0,0,0,0.08)"}),
                        html.P("LECTURA DEL MAPA", style=STYLE_LABEL),
                        *[
                            html.Div(style={"display": "flex", "alignItems": "center",
                                           "marginBottom": "4px"}, children=[
                                html.Div(style={"width": "14px", "height": "14px",
                                               "backgroundColor": color,
                                               "borderRadius": "3px", "marginRight": "8px"}),
                                html.P(label, style={"fontSize": "11px",
                                                     "color": COLOR_TEXT_MID, "margin": 0}),
                            ])
                            for color, label in [
                                ("#FFB300", "Brecha baja — mayor equidad"),
                                ("#4a7ab5", "Brecha media — alerta territorial"),
                                ("#0a1628", "Brecha alta — desigualdad estructural"),
                            ]
                        ],
                    ]),
                ], md=4),
            ]),
        ]),

        # PARRAFO CONCLUSION
        dbc.Container(fluid=True, style={"padding": "16px 32px 8px"}, children=[
            html.Div(style={**STYLE_CARD, "borderLeft": f"4px solid {COLOR_RURAL}"}, children=[
                html.P("REFLEXION SOBRE LOS DATOS", style=STYLE_LABEL),
                html.P([
                    "El acceso a ",
                    html.Strong("fuentes de agua mejorada"),
                    " es uno de los 15 indicadores del IPM del DANE. Su ausencia es una privacion "
                    "que afecta la salud, la dignidad y las oportunidades de miles de hogares. "
                    "Lo que este mapa revela es que la desigualdad en este acceso sigue un patron "
                    "territorial sistematico: los departamentos de la Amazonia, la Orinoquia y el "
                    "Pacifico concentran las brechas mas profundas. Entre 2018 y 2025 la brecha "
                    "nacional se redujo, pero de forma insuficiente y desigual. ",
                    html.Strong(
                        f"En 2025, un hogar rural colombiano tiene en promedio "
                        f"{nac[2025]['rural']:.1f}% de probabilidad de carecer de agua mejorada, "
                        f"frente al {nac[2025]['cabeceras']:.1f}% en cabeceras: "
                        f"una brecha de {nac[2025]['brecha']:.1f} pp que el tiempo no ha logrado cerrar."
                    ),
                ], style={"fontSize": "13px", "color": COLOR_TEXT_MID, "lineHeight": "1.8"}),
            ]),
        ]),

        # FUENTES
        dbc.Container(fluid=True, style={"padding": "8px 32px 24px"}, children=[
            html.Hr(style={"borderColor": "rgba(0,0,0,0.1)"}),
            html.Small([
                html.Strong("Fuente: "),
                "DANE — Encuesta Nacional de Calidad de Vida (ECV) 2018-2025. "
                "Indicador: Privaciones por hogar segun variable, hoja IPM_Indicadores_Departamento. "
                "Cartografia: Marco Geoestadistico Nacional DANE 2024 (MGN2024_DPTO_POLITICO). "
                "Nota: Los valores de 2020 y 2021 incorporan ajustes metodologicos del DANE "
                "derivados de la pandemia COVID-19. Elaboracion propia con Python, Dash y Plotly.",
            ], style={"color": "#999", "fontSize": "10px", "lineHeight": "1.5"}),
        ]),

        # Store y Modal
        dcc.Store(id="selected-dept-store"),
        dbc.Modal(
            id="modal-dept",
            size="lg", is_open=False, backdrop=True, scrollable=True,
            children=[
                dbc.ModalHeader(
                    html.H5(id="modal-titulo",
                            style={"fontWeight": "800", "color": COLOR_TEXT_DARK,
                                   "fontSize": "18px"}),
                    close_button=True,
                ),
                dbc.ModalBody([
                    html.P(id="modal-subtitulo",
                           style={"fontSize": "11px", "color": COLOR_TEXT_MID,
                                  "letterSpacing": "1px", "textTransform": "uppercase",
                                  "marginBottom": "4px"}),
                    dcc.Graph(id="chart-modal",
                              config={"displayModeBar": False},
                              style={"height": "340px"}),
                    html.Hr(style={"margin": "12px 0"}),
                    html.P("SITUACION DEL DEPARTAMENTO",
                           style={**STYLE_LABEL, "marginBottom": "6px"}),
                    html.Div(id="modal-parrafo"),
                ]),
            ],
        ),
    ],
)

print("Layout definido OK")

## Celda 6 — Callbacks de interactividad

Dos callbacks:
1. **`update_main_charts`**: actualiza mapa y grafico nacional cuando cambia el slider de ano
2. **`open_dept_modal`**: cuando el usuario hace click en un departamento del mapa,
   abre el modal con la serie temporal y parrafo reflexivo de ese departamento


In [ ]:
@app.callback(
    Output("mapa-agua",     "figure"),
    Output("chart-nacional", "figure"),
    Input("year-slider",    "value"),
    Input("selected-dept-store", "data"),
)
def update_main_charts(year, selected_dept):
    """Rebuild choropleth and national chart when year or department selection changes."""
    fig_map = build_map(gdf, year, selected_dpto=selected_dept)
    fig_nac = build_national_chart(nac, year)
    return fig_map, fig_nac


@app.callback(
    Output("selected-dept-store", "data"),
    Output("modal-dept",      "is_open"),
    Output("modal-titulo",    "children"),
    Output("modal-subtitulo", "children"),
    Output("chart-modal",     "figure"),
    Output("modal-parrafo",   "children"),
    Input("mapa-agua",        "clickData"),
    State("year-slider",      "value"),
    prevent_initial_call=True,
)
def open_dept_modal(click_data, year):
    """Open modal with department time series when user clicks on the map."""
    if not click_data:
        return None, False, "", "", go.Figure(), []

    # Indice del departamento clickeado (el mapa usa indices enteros como locations)
    point_idx = click_data["points"][0]["location"]
    row_gdf   = gdf.iloc[point_idx]
    dept_name = str(row_gdf.get("Departamento", row_gdf.get("dpto_norm", "")))

    # Buscar fila correspondiente en el DataFrame de agua
    dept_norm = normalize_name(dept_name)
    match     = agua[agua["Departamento_norm"] == dept_norm]

    if match.empty:
        return dept_name, False, "", "", go.Figure(), []

    dept_row  = match.iloc[0]
    titulo    = f"{dept_row['Departamento']} — Acceso a agua mejorada 2018-2025"
    subtitulo = (f"Brecha 2025: {dept_row['Brecha_2025']:.1f} pp  |  "
                 f"Cabeceras: {dept_row['Cabeceras_2025']:.1f}%  |  "
                 f"Rural: {dept_row['Rural_2025']:.1f}%")
    fig_modal = build_dept_modal_chart(dept_row, dept_row["Departamento"])
    parrafo   = generate_dept_paragraph(dept_row, dept_row["Departamento"])

    return dept_name, True, titulo, subtitulo, fig_modal, parrafo


print("Callbacks registrados OK")

## Celda 7 — Exportacion a HTML y ejecucion

### Como visualizar el resultado

**Opcion 1 — Dashboard interactivo completo (VS Code):**
```bash
python agua_brecha_app.py
# Luego abrir http://localhost:8050 en el navegador
```

**Opcion 2 — HTML estatico (entrega):**
La funcion `export_static_html()` genera `agua_brecha_mapa.html` con el mapa
y el grafico nacional en un solo archivo HTML descargable que funciona sin servidor.
La interactividad de click (modal) no esta disponible en la version estatica.

**Opcion 3 — Jupyter/Colab:**
Ejecutar la celda de abajo. El servidor corre en el puerto 8050.
En Colab puede requerirse `jupyter-dash` y su metodo `run_server(mode='inline')`.


In [ ]:
def export_static_html(output_path: str = "agua_brecha_mapa.html"):
    """
    Export a static HTML combining the choropleth and national trend chart.
    Useful for delivery when a Dash server is not available.
    Note: modal interactivity (click on department) requires the Dash server.
    """
    import plotly.io as pio
    from plotly.subplots import make_subplots

    fig_map = build_map(gdf, 2025)
    fig_nac = build_national_chart(nac, 2025)

    combined = make_subplots(
        rows=1, cols=2,
        column_widths=[0.65, 0.35],
        specs=[[{"type": "choropleth"}, {"type": "xy"}]],
        subplot_titles=[
            "Brecha en acceso a agua mejorada por departamento (2025)",
            "Evolucion nacional 2018-2025",
        ],
    )

    for trace in fig_map.data:
        combined.add_trace(trace, row=1, col=1)
    for trace in fig_nac.data:
        combined.add_trace(trace, row=1, col=2)

    combined.update_geos(fitbounds="locations", visible=False, row=1, col=1)
    combined.update_layout(
        title=dict(
            text=("<b>El campo sigue sin agua: la desigualdad que divide a Colombia</b><br>"
                  "<span style='font-size:12px;color:#4a5568;font-style:italic;'>"
                  "Brecha en % de hogares sin acceso a fuente de agua mejorada — "
                  "Cabeceras vs. Rural disperso — DANE ECV 2018-2025</span>"),
            x=0.02, font=dict(size=16),
        ),
        paper_bgcolor=COLOR_PAGE_BG,
        plot_bgcolor=COLOR_PAGE_BG,
        height=620,
        margin=dict(l=20, r=20, t=90, b=60),
        showlegend=True,
        legend=dict(orientation="h", y=-0.08, font=dict(size=10)),
        annotations=list(combined.layout.annotations) + [
            dict(
                xref="paper", yref="paper", x=0.02, y=-0.14,
                text=("Fuente: DANE — ECV 2018-2025. Cartografia: MGN2024 DANE. "
                      "Color del mapa = brecha (Rural - Cabeceras). "
                      "Dorado = menor brecha. Azul oscuro = mayor desigualdad."),
                showarrow=False, font=dict(size=9, color="#999"), align="left",
            )
        ],
    )

    pio.write_html(combined, file=output_path, full_html=True,
                  include_plotlyjs=True,
                  config={"displayModeBar": True, "scrollZoom": False})
    print(f"HTML estatico exportado -> {output_path}")


# Exportar HTML estatico
export_static_html("agua_brecha_mapa.html")

# Iniciar dashboard interactivo
# En VS Code ejecutar directamente: python agua_brecha_app.py
# En Colab comentar la linea de abajo y usar run_server(mode='inline')
app.run(debug=False, host="0.0.0.0", port=8050)